In [14]:
#AttributeError: module 'gradio' has no attribute 'blocks' 에러 발생 시
#pip install --upgrade gradio

In [1]:
import pandas as pd
import numpy as np

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [15]:
deposit = pd.read_csv('/content/drive/MyDrive/Dataset/deposit.csv')
monthly = pd.read_csv('/content/drive/MyDrive/Dataset/monthly.csv')

In [16]:
deposit.head()

,행정동명,보증금(만원),임대면적(㎡),행정동_코드,medical,education,shopping,bank,gov,culture,intercity,intracity,자치구명,crime_rate,clearance_rate,crime_grade,lat,lon
0,가락2동,65000.0,59.968,11710632,17,7,0,6,3,0,0,23,송파구,0.064020,0.732509,DANGER,37.498698,127.126640
1,가리봉동,15000.0,30.330,11530595,6,2,0,0,0,23,0,8,구로구,0.041797,0.729130,NORMAL,37.482533,126.889718
2,가산동,20050.0,25.630,11545510,22,2,0,25,10,18,0,114,금천구,0.024957,0.847794,SAFE,37.476876,126.891779
3,가양1동,34000.0,49.500,11500603,20,6,0,13,5,3,0,74,강서구,0.049592,0.851048,DANGER,37.569497,126.844729
4,가회동,24750.0,43.480,11110600,1,6,0,3,5,0,0,22,종로구,0.034212,1.174684,NORMAL,37.580097,126.984784


In [17]:
monthly.head()

,행정동명,보증금(만원),임대료(만원),임대면적(㎡),행정동_코드,medical,education,shopping,bank,gov,culture,intercity,intracity,자치구명,crime_rate,clearance_rate,crime_grade,lat,lon
0,가락2동,14954.0,61.0,43.070,11710632,17,7,0,6,3,0,0,23,송파구,0.064020,0.732509,DANGER,37.498698,127.126640
1,가리봉동,1000.0,47.0,22.500,11530595,6,2,0,0,0,23,0,8,구로구,0.041797,0.729130,NORMAL,37.482533,126.889718
2,가산동,1000.0,55.0,17.690,11545510,22,2,0,25,10,18,0,114,금천구,0.024957,0.847794,SAFE,37.476876,126.891779
3,가양1동,3000.0,60.0,34.440,11500603,20,6,0,13,5,3,0,74,강서구,0.049592,0.851048,DANGER,37.569497,126.844729
4,가회동,4000.0,67.5,43.625,11110600,1,6,0,3,5,0,0,22,종로구,0.034212,1.174684,NORMAL,37.580097,126.984784


In [18]:
crime_map = {
    'SAFE': 3,
    'NORMAL': 2,
    'DANGER': 1
}

deposit['crime_score'] = deposit['crime_grade'].map(crime_map)
monthly['crime_score'] = monthly['crime_grade'].map(crime_map)

In [19]:
deposit[['crime_grade','crime_score']].head()

,crime_grade,crime_score
0,DANGER,1
1,NORMAL,2
2,SAFE,3
3,DANGER,1
4,NORMAL,2


In [20]:
import requests

ip = requests.get("https://api.ipify.org").text
print(ip)

34.181.144.38


In [25]:
#API Key
ODSAY_API_KEY = "{YOUR ODSAY_API_KEY}"
KAKAO_API_KEY = "{YOUR KAKAO_API_KEY}"

In [26]:
def address_to_coord(address):

    url = "https://dapi.kakao.com/v2/local/search/address.json"

    headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
    params = {"query": address}

    response = requests.get(
        url,
        headers=headers,
        params=params)

    result = response.json()

    if len(result["documents"]) == 0:
        return None, None


    x = float(result["documents"][0]["x"])
    y = float(result["documents"][0]["y"])

    return x, y

In [27]:
def get_transit_time(
    start_lon,
    start_lat,
    end_lon,
    end_lat):

    url = "https://api.odsay.com/v1/api/searchPubTransPathT"

    params = {
        "SX": start_lon,
        "SY": start_lat,
        "EX": end_lon,
        "EY": end_lat,
        "apiKey": ODSAY_API_KEY}

    try:
        response = requests.get(
            url,
            params=params,
            timeout=10)

        data = response.json()
        path = data["result"]["path"][0]
        return path["info"]["totalTime"]

    except Exception:
        return None

In [28]:
import gradio as gr

def recommend_house(
    house_type,
    deposit_budget,
    monthly_budget,
    priority1,
    priority2,
    priority3,
    safety_option,
    use_commute,
    workplace,
    max_commute):

    #전/월세 데이터 선택
    if house_type == "전세":

        candidate = deposit.copy()

        candidate = candidate[
            candidate["보증금(만원)"] <= deposit_budget
        ]

    else:

        candidate = monthly.copy()

        candidate = candidate[
            (candidate["보증금(만원)"] <= deposit_budget)
            &
            (candidate["임대료(만원)"] <= monthly_budget)
        ]

    if len(candidate) == 0:
        return pd.DataFrame(
            {"결과": ["조건에 맞는 매물이 없습니다."]}
        )

    #중요 시설 선택

    facility_map = {
    "의료": "medical",
    "교육": "education",
    "쇼핑": "shopping",
    "은행": "bank",
    "관공서": "gov",
    "문화": "culture",
    "시외 교통": "intercity",
    "시내 교통": "intracity"
    }

    candidate["facility_score"] = (
        candidate[facility_map[priority1]] * 3
        + candidate[facility_map[priority2]] * 2
        + candidate[facility_map[priority3]] * 1
    )

    #정규화
    candidate["facility_norm"] = (
        candidate["facility_score"]
        - candidate["facility_score"].min()
    ) / (
        candidate["facility_score"].max()
        - candidate["facility_score"].min()
        + 1e-9)

    candidate["crime_norm"] = (
        candidate["crime_score"]
        - candidate["crime_score"].min()
    ) / (
        candidate["crime_score"].max()
        - candidate["crime_score"].min()
        + 1e-9)

    #최종 점수 반환
    if safety_option:
        candidate["score"] = (
            candidate["facility_norm"]
            + candidate["crime_norm"]) / 2

    else:
        candidate["score"] = candidate["facility_norm"]

    #이동 시간 고려하는 경우
    if use_commute and workplace:

        work_lon, work_lat = address_to_coord(
            workplace)

        candidate = candidate.sort_values("score", ascending=False).head(20)

        candidate["commute_time"] = candidate.apply(
            lambda row: get_transit_time(
                row["lon"],
                row["lat"],
                work_lon,
                work_lat),axis=1)

        candidate = candidate.dropna(
            subset=["commute_time"])

        candidate = candidate[
            candidate["commute_time"]
            <= max_commute]

    else:
        candidate["commute_time"] = None

    candidate = candidate.sort_values(
        "score",
        ascending=False
    )

    cols = [
        "행정동명",
        "보증금(만원)",
        "crime_grade",
        "facility_score",
        "commute_time",
        "score"
    ]

    if house_type == "월세":
        cols.insert(2, "임대료(만원)")

    return candidate[cols].head(3)

#gradio
with gr.Blocks() as demo:

    gr.Markdown("# 서울 자취방 추천 시스템")

    house_type = gr.Radio(
        ["전세", "월세"],
        value="전세",
        label="거래 유형")

    deposit_input = gr.Number(
        label="보증금(만원)")

    monthly_input = gr.Number(
        label="월세(만원)",
        visible=False)

    priority1 = gr.Dropdown(
    ["의료","교육","쇼핑","은행","관공서","문화","시외 교통","시내 교통"],
    label="1순위 중요 요소"
    )

    priority2 = gr.Dropdown(
        ["의료","교육","쇼핑","은행","관공서","문화","시외 교통","시내 교통"],
        label="2순위 중요 요소"
    )

    priority3 = gr.Dropdown(
        ["의료","교육","쇼핑","은행","관공서","문화","시외 교통","시내 교통"],
        label="3순위 중요 요소"
    )

    safety_option = gr.Checkbox(
        label="안전을 중요하게 생각함"
    )

    use_commute = gr.Checkbox(
        label="이동 시간 고려")

    workplace = gr.Textbox(
        label="방문 장소")

    max_commute = gr.Slider(
        10,
        120,
        value=30,
        step=5,
        label="최대 이동 시간(분)")

    output = gr.Dataframe(
        label="추천 결과")

    btn = gr.Button("추천 받기")

    def update_house_type(house_type):

        if house_type == "월세":
            return gr.update(visible=True)

        return gr.update(visible=False)

    house_type.change(
        update_house_type,
        inputs=house_type,
        outputs=monthly_input)

    btn.click(
        fn=recommend_house,
        inputs=[
            house_type,
            deposit_input,
            monthly_input,
            priority1,
            priority2,
            priority3,
            safety_option,
            use_commute,
            workplace,
            max_commute],outputs=output)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6b6d7fe8c7a795ee73.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
